<a href="https://colab.research.google.com/github/fboldt/aulas-am-bsi/blob/main/aula20a_sele%C3%A7%C3%A3o_de_caracter%C3%ADsticas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

arrhythmia = fetch_openml('arrhythmia', version=1)
X, y = arrhythmia.data, arrhythmia.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
imputer = SimpleImputer(strategy='mean')
X_train = imputer.fit_transform(X_train)
X_test = imputer.transform(X_test)
print(X_train.shape)

(361, 279)


In [2]:
from sklearn.linear_model import RidgeClassifier
model = RidgeClassifier()

def evaluate_model(model, data):
    X_train, X_test, y_train, y_test = data
    model.fit(X_train, y_train)
    return model.score(X_test, y_test)

data = (X_train, X_test, y_train, y_test)
print(evaluate_model(model, data))

0.5824175824175825


In [13]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

normalization = StandardScaler()
model_std = make_pipeline(normalization, model)
print(evaluate_model(model_std, data))

0.5164835164835165


#PCA

In [19]:
from sklearn.decomposition import PCA

pca = PCA(n_components=0.99)
model_pca = make_pipeline(pca, model)
print(evaluate_model(model_pca, data))

0.7032967032967034


In [29]:
n_feats = model_pca.steps[0][1].n_components_
print(n_feats)

63


In [26]:
model_std_pca = make_pipeline(normalization, pca, model)
print(evaluate_model(model_std_pca, data))

0.6593406593406593


In [27]:
model_pca_std = make_pipeline(pca, normalization, model)
print(evaluate_model(model_pca_std, data))

0.7032967032967034


# Feature Selection
## Ranking
### Filter

In [30]:
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin

def variance_score(X, y=None):
    return np.var(X, axis=0)[0]

def feature_ranking(X, y=None, score_function=variance_score):
    scores = []
    for f in range(X.shape[1]):
        score = score_function(X[:, f:f+1], y)
        scores.append(score)
    return np.argsort(scores)[::-1]

class VarianceThreshold(BaseEstimator, TransformerMixin):
    def __init__(self, n_feats=None, score_function=variance_score):
        self.n_feats = n_feats
        self.score_function = score_function
    def fit(self, X, y=None):
        if self.n_feats is None:
            self.feats = X.shape[1]
        ranking = feature_ranking(X, y, self.score_function)
        self.feats = ranking[:self.n_feats]
        return self
    def transform(self, X, y=None):
        return X[:, self.feats]

feat_rank = VarianceThreshold(n_feats=n_feats)
feat_rank.fit(X_train, y_train)
print(feat_rank.feats)

[ 10  13   9   5   2   7 248  12 258   6 247  11  56  87 238  51   8  41
 237 268 257  89  40  64  65  77 149  39  99  29 227  76  17  53 278  16
  44 178  92  63  28 188 101 228 137  52   3 125   0 267 218 187 113   4
  75  68 111 148 177 100 217  88 277]


In [31]:
model_fr = make_pipeline(feat_rank, model)
print(evaluate_model(model_fr, data))

0.6703296703296703


In [37]:
class HybridRanking(BaseEstimator, TransformerMixin):
    def __init__(self, model, score_function=variance_score):
        self.model = model
        self.score_function = score_function
    def fit(self, X, y=None):
        X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
        ranking = feature_ranking(X_tr, y_tr, self.score_function)
        best_score = -np.inf
        best_n = 0
        for n in range(1, X_tr.shape[1]+1):
            feats = ranking[:n]
            model = self.model.fit(X_tr[:, feats], y_tr)
            score = model.score(X_val[:, feats], y_val)
            if score > best_score:
                best_score = score
                best_n = n
        self.feats = ranking[:best_n]
        return self
    def transform(self, X, y=None):
        return X[:, self.feats]

hybr_rank = HybridRanking(model)
hybr_rank.fit(X_train, y_train)
print(len(hybr_rank.feats))
print(hybr_rank.feats)

84
[ 10  13   5   2   9 248   7  12 258   6 247  11  56  87 237 257   8 268
 238  41  51  64  89  40  99  77  39  65 149 227  29  17  76 278  53 178
  63  16  44 188 101  92 228 137  28  52 267 218   0 111 187 113 125  68
  75   3 148   4 100  14  88 217 277 177 167  80 207 198 168 112  90 208
 136 140 197 104 152  54  20 147 135  27 123  15]


In [38]:
model_hr = make_pipeline(hybr_rank, model)
model_hr.fit(X_train, y_train)
print(f"Resubstituition: {model_hr.score(X_train, y_train)}")#
print(f"Holdout: {evaluate_model(model_hr, data)}")

Resubstituition: 0.8282548476454293
Holdout: 0.6703296703296703


In [40]:
import numpy as np

TMP = np.array([
    [1, 2, 3],
    [0, 1, 1],
    [4, 5, 6],
    [1, 4, 1],
    [7, 8, 9],
    [6, 1, 8],
])

def pairwise_distances(X1, X2=None):
  if X2 is None:
    X2 = X1
  return np.linalg.norm(X1[:, None, :] - X2[None, :, :], axis=-1)

print(pairwise_distances(TMP))

[[ 0.          2.44948974  5.19615242  2.82842712 10.39230485  7.14142843]
 [ 2.44948974  0.          7.54983444  3.16227766 12.72792206  9.21954446]
 [ 5.19615242  7.54983444  0.          5.91607978  5.19615242  4.89897949]
 [ 2.82842712  3.16227766  5.91607978  0.         10.77032961  9.11043358]
 [10.39230485 12.72792206  5.19615242 10.77032961  0.          7.14142843]
 [ 7.14142843  9.21954446  4.89897949  9.11043358  7.14142843  0.        ]]


In [41]:
for f in range(TMP.shape[1]):
  print(f"Feature: {f}")
  dists = pairwise_distances(TMP[:, f:f+1])
  print(dists)
  print(f"mean distance: {dists.mean()}")

Feature: 0
[[0. 1. 3. 0. 6. 5.]
 [1. 0. 4. 1. 7. 6.]
 [3. 4. 0. 3. 3. 2.]
 [0. 1. 3. 0. 6. 5.]
 [6. 7. 3. 6. 0. 1.]
 [5. 6. 2. 5. 1. 0.]]
mean distance: 2.9444444444444446
Feature: 1
[[0. 1. 3. 2. 6. 1.]
 [1. 0. 4. 3. 7. 0.]
 [3. 4. 0. 1. 3. 4.]
 [2. 3. 1. 0. 4. 3.]
 [6. 7. 3. 4. 0. 7.]
 [1. 0. 4. 3. 7. 0.]]
mean distance: 2.7222222222222223
Feature: 2
[[0. 2. 3. 2. 6. 5.]
 [2. 0. 5. 0. 8. 7.]
 [3. 5. 0. 5. 3. 2.]
 [2. 0. 5. 0. 8. 7.]
 [6. 8. 3. 8. 0. 1.]
 [5. 7. 2. 7. 1. 0.]]
mean distance: 3.5555555555555554


In [42]:
def intra_class_mean_distance(X, y):
  mean_dist = 0
  labels = np.unique(y)
  for label in labels:
    X_label = X[y==label]
    dists = pairwise_distances(X_label)
    mean_dist += dists.mean()
  return mean_dist / len(labels)

y_tmp = np.array([0,1,0,1,0,1])
print(intra_class_mean_distance(TMP, y_tmp))

4.697429487492455


In [43]:
def inter_class_mean_distance(X, y):
  mean_dist = 0
  labels = np.unique(y)
  for label in labels:
    in_class = X[y==label]
    out_class = X[y!=label]
    dists = pairwise_distances(in_class, out_class)
    mean_dist += dists.mean()
  return mean_dist / len(labels)

print(inter_class_mean_distance(TMP, y_tmp))

6.824879900464295


In [46]:
def distance_score(X, y):
  return inter_class_mean_distance(X, y) / (intra_class_mean_distance(X, y)+1e-8)

print(distance_score(TMP, y_tmp))

1.452896718110937


In [47]:
feat_rank_dist = HybridRanking(model, score_function=distance_score)
feat_rank_dist.fit(X_train, y_train)
print(len(feat_rank_dist.feats))
print(feat_rank_dist.feats)

75
[ 48 150  23  37  47  59 153  45 155  71  73  85  97  84 130 273  43 113
   5  16 184   4 243 111 148 237   8  41 229 227  39  18 114  52  56 277
  27  14 275 265  90 170  20 247 236  51   7  66 112  95 163  28 257  63
   6  68 197  32 246 272 182 177  17 268 198   1 125 166 258 171  92 100
  99  13 190]


In [50]:
model_hrd = make_pipeline(feat_rank_dist, model)
print(evaluate_model(model_hrd, data))

0.6703296703296703


### Wrapper

In [51]:
def wrapper_score(X, y):
  X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
  model.fit(X_tr, y_tr)
  return model.score(X_val, y_val)

feat_rank_rapper = HybridRanking(model, score_function=wrapper_score)
feat_rank_rapper.fit(X_train, y_train)
print(len(feat_rank_rapper.feats))
print(feat_rank_rapper.feats)

76
[ 90  92 111 166 240 266 276 196 148 237 247 176 152 210 170 191 140   4
 172 142  20 220  16 121 113 136  26 256 123  94  88  89  73  71  87  72
  86  85  84  83  81  74  75  80 110  78  76  77  82 124 122 120 119 118
 116 115 114 112 143 141 109 108 107  99 105 104 103 102 101 100  70  98
  97  96 127 125]


In [52]:
model_frw = make_pipeline(feat_rank_rapper, model)
print(evaluate_model(model_frw, data))

0.6263736263736264
